# 은행 마케팅 캠페인 데이터 전처리

- 원본 데이터: 56,373건
- duration 이상치 제거: 84건
- campaign 이상치 제거: 3,418건
- 최종 데이터: 41,860건


In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import chi2_contingency, ttest_ind, mannwhitneyu
import warnings
warnings.filterwarnings('ignore')
 
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False
 
 
df = pd.read_csv('Bank_Target_Marketing.csv')
df.head()

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 56373 entries, 0 to 56372
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        56373 non-null  int64 
 1   job        56373 non-null  object
 2   marital    56373 non-null  object
 3   education  56373 non-null  object
 4   default    56373 non-null  object
 5   balance    56373 non-null  int64 
 6   housing    56373 non-null  object
 7   loan       56373 non-null  object
 8   contact    56373 non-null  object
 9   day        56373 non-null  int64 
 10  month      56373 non-null  object
 11  duration   56373 non-null  int64 
 12  campaign   56373 non-null  int64 
 13  pdays      56373 non-null  int64 
 14  previous   56373 non-null  int64 
 15  poutcome   56373 non-null  object
 16  deposit    56373 non-null  object
dtypes: int64(7), object(10)
memory usage: 7.3+ MB


## 타겟 변수 생성

`deposit` 컬럼이 'yes'/'no' 문자열로 되어있어
통계 분석에 사용하기 위해 0/1로 변환

In [3]:
df['deposit_yn'] = (df['deposit'] == 'yes').astype(int)

print(df[['deposit','deposit_yn']].value_counts().sort_index())

deposit  deposit_yn
no       0             45795
yes      1             10578
Name: count, dtype: int64


## 신규 / 기존 고객 분리 (previous=0이면 이번 캠페인이 처음)

`previous` 컬럼은 이전 캠페인 접촉 횟수
- `previous = 0`: 이번 캠페인이 처음인 신규 고객
- `previous > 0`: 이전 캠페인에 접촉된 기존 고객

두 그룹은 행동 패턴이 다를 수 있어 분리하여 분석함

In [4]:
new  = df[df['previous'] == 0].copy()
prev = df[df['previous'] >  0].copy()
 
print(f"신규: {len(new):,}명 ({len(new)/len(df)*100:.1f}%)")
print(f"기존: {len(prev):,}명 ({len(prev)/len(df)*100:.1f}%)")
print(f"신규 가입률: {new['deposit_yn'].mean()*100:.1f}%  |  기존 가입률: {prev['deposit_yn'].mean()*100:.1f}%")

신규: 45,278명 (80.3%)
기존: 11,095명 (19.7%)
신규 가입률: 14.9%  |  기존 가입률: 34.3%


## 파생변수 생성

| 파생변수 | 원본 변수 | 기준 |
|----------|-----------|------|
| age_grp | age | 29세 이하 / 30대 / 40대 / 50대 / 60세 이상 |
| bal_grp | balance | 하위 33% / 중간 / 상위 33% |
| month_n | month | 월 문자열 → 숫자 변환 |

In [5]:
def age_grp(age):
    if age < 30:   return '29세이하'
    elif age < 40: return '30대'
    elif age < 50: return '40대'
    elif age < 60: return '50대'
    else:           return '60세이상'
 
age_order = ['29세이하', '30대', '40대', '50대', '60세이상']
 
for d in [df, new, prev]:
    d['age_grp'] = d['age'].apply(age_grp)

print(df['age_grp'].value_counts().reindex(age_order))

age_grp
29세이하     6824
30대      22407
40대      14283
50대      10295
60세이상     2564
Name: count, dtype: int64


In [6]:
q1, q2 = df['balance'].quantile([0.33, 0.67])
 
def bal_grp(x):
    if x <= q1:  return '저소득'
    elif x <= q2: return '중산층'
    else:          return '고소득'
 
for d in [df, new, prev]:
    d['bal_grp'] = d['balance'].apply(bal_grp)

print(f"33분위수(저소득 기준): {q1:.0f}")
print(f"67분위수(고소득 기준): {q2:.0f}")
print(df['bal_grp'].value_counts())

33분위수(저소득 기준): 182
67분위수(고소득 기준): 1007
bal_grp
중산층    19165
저소득    18608
고소득    18600
Name: count, dtype: int64


In [7]:
mmap = {'jan':1,'feb':2,'mar':3,'apr':4,'may':5,'jun':6,
        'jul':7,'aug':8,'sep':9,'oct':10,'nov':11,'dec':12}
mlabel = {'jan':'Jan','feb':'Feb','mar':'Mar','apr':'Apr','may':'May','jun':'Jun',
          'jul':'Jul','aug':'Aug','sep':'Sep','oct':'Oct','nov':'Nov','dec':'Dec'}
 
for d in [df, new, prev]:
    d['month_n'] = d['month'].map(mmap)
    d['month_l'] = d['month'].map(mlabel)

print(df[['month','month_n','month_l']].drop_duplicates().sort_values('month_n').to_string(index=False))

month  month_n month_l
  jan        1     Jan
  feb        2     Feb
  mar        3     Mar
  apr        4     Apr
  may        5     May
  jun        6     Jun
  jul        7     Jul
  aug        8     Aug
  sep        9     Sep
  oct       10     Oct
  nov       11     Nov
  dec       12     Dec


## 이상치 제거


### duration (통화 시간)
- 2000초(약 33분) 초과 값을 이상치로 판단하여 제거
- 근거: 현실적으로 33분 이상의 마케팅 통화는 일반적이지 않으며, 데이터 입력 오류 가능성

### campaign (연락 횟수)
- IQR 방식으로 상한선 초과 값을 제거
- 근거: 지나치게 많은 연락은 고객 거부감으로 이어져 일반적 패턴과 다를 수 있음

In [8]:
before = len(new)
new_d  = new[new['duration'] <= 2000].copy()
after  = len(new_d)

print(f"제거 전: {before:,}명")
print(f"제거 후: {after:,}명  (제거: {before-after}명, {(before-after)/before*100:.1f}%)")
print(f"duration 분포 (제거 후):")
print(new_d['duration'].describe().apply(lambda x: f"{x:.1f}").to_string())

제거 전: 45,278명
제거 후: 45,194명  (제거: 84명, 0.2%)
duration 분포 (제거 후):
count    45194.0
mean       276.1
std        269.9
min          0.0
25%        105.0
50%        187.0
75%        343.0
max       1994.0


In [9]:
q3  = new['campaign'].quantile(0.75)
iqr = q3 - new['campaign'].quantile(0.25)
upper = q3 + 1.5 * iqr
 
before_c = len(new)
new_c    = new[new['campaign'] <= upper].copy()
after_c  = len(new_c)
 
print(f"제거 전: {before_c:,}명")
print(f"제거 후: {after_c:,}명  (제거: {before_c-after_c}명, {(before_c-after_c)/before_c*100:.1f}%)")
print(f"campaign 분포 (제거 후):")
print(new_c['campaign'].describe().apply(lambda x: f"{x:.1f}").to_string())


제거 전: 45,278명
제거 후: 41,860명  (제거: 3418명, 7.5%)
campaign 분포 (제거 후):
count    41860.0
mean         2.2
std          1.3
min          1.0
25%          1.0
50%          2.0
75%          3.0
max          6.0


In [10]:
print("최종 shape:", new_c.shape)
new_c.describe()

최종 shape: (41860, 22)


,age,balance,day,duration,campaign,pdays,previous,deposit_yn,month_n
count,41860.000000,41860.000000,41860.000000,41860.000000,41860.000000,41860.0,41860.0,41860.000000,41860.000000
mean,40.937028,1342.458266,15.796464,286.052222,2.169374,-1.0,0.0,0.154611,6.175012
std,10.708794,3030.349274,8.311530,288.798137,1.331694,0.0,0.0,0.361537,2.293789
min,18.000000,-8019.000000,1.000000,0.000000,1.000000,-1.0,0.0,0.000000,1.000000
25%,32.000000,66.000000,8.000000,111.000000,1.000000,-1.0,0.0,0.000000,5.000000
50%,39.000000,434.000000,16.000000,193.000000,2.000000,-1.0,0.0,0.000000,6.000000
75%,49.000000,1400.000000,21.000000,351.000000,3.000000,-1.0,0.0,0.000000,7.000000
max,95.000000,102127.000000,31.000000,4918.000000,6.000000,-1.0,0.0,1.000000,12.000000


In [11]:
new_c.to_csv('new_customers_cleaned.csv', index=False)
prev.to_csv('prev_customers_cleaned.csv', index=False)

## 전처리 완료
원본 56,373건에서 이상치를 제거하여
최종 41,860건의 데이터를 저장.
- duration 이상치 84건 제거
- campaign 이상치 3,418건 제거 